# 08: Inference on the fresh confirmation set (label-free)

Runs all 33 existing checkpoints (11 architectures x 3 seeds) over the 15,794 fresh images
(7,897 patients, never used in training, validation or test). **No retraining, no evaluation,
no metrics computed.** It only writes float64 predictions, so it can safely run before the
v3 lock is written.

- Strict checkpoint loading into weight-free skeletons; any mismatch fails loudly.
- Same eval transform as training (resize 256, centre crop 224, ImageNet normalisation).
- Output: `eyepacs_predictions_fresh_<model>[_seedK]_v2.csv` in `03_predictions`.
- Resumable: skips any model already done.

A100 recommended. About 1.5 to 2 hours, including ~10 GB of checkpoint downloads.
Needs the `HF_TOKEN` secret (for the RETFound config only). Run cells 1-4.

In [2]:
# Cell 1: auth, deps, Drive helpers
!pip -q install timm transformers
import io, os, json, time, zipfile, glob, datetime
import numpy as np, pandas as pd, torch
import torch.nn as nn
from google.colab import auth, userdata
auth.authenticate_user()
from huggingface_hub import login
login(userdata.get('HF_TOKEN'))
from googleapiclient.discovery import build as gbuild
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
svc = gbuild('drive', 'v3')

F_CHECKPOINTS = '1DAAXJ2fjk2zAzx7PpGaWsk4BXudeA5Le'
F_PREDICTIONS = '1nCloJ5o9veZb2xJL_2EjdVHQXN31jy4w'
F_SPLITS      = '13x_CS7N_UasGRSuNYMyD5To2RTYjL-EP'
F_FRESH_IMG   = '15mufJLdRXNtZFSHQ_kFHIwmON8ewrPAX'
F_AUDIT       = '1zuvfcxC3umHAq_uf7Vx9omMEgUEuQhTt'
RETFOUND_REPO = 'iszt/RETFound_mae_meh'
NUM_WORKERS   = min(8, os.cpu_count())

ARCHS = ['densenet121_binary','densenet121_5class','efficientnet_b3_binary','efficientnet_b3_5class',
         'resnet50_binary','resnet50_5class','vit_base_binary','vit_base_5class',
         'retfound_binary','retfound_5class','retfound_adversarial']
SEEDS = [0, 1, 2]
def tag(k): return '' if k == 0 else f'_seed{k}'

def dl(fid, path):
    with io.FileIO(path, 'wb') as fh:
        d = MediaIoBaseDownload(fh, svc.files().get_media(fileId=fid), chunksize=64*1024*1024); done = False
        while not done: _, done = d.next_chunk()
def find_id(name, folder):
    r = svc.files().list(q=f"'{folder}' in parents and name='{name}' and trashed=false",
                         fields='files(id)').execute().get('files', [])
    return r[0]['id'] if r else None
def list_folder(folder):
    out, tok = [], None
    while True:
        r = svc.files().list(q=f"'{folder}' in parents and trashed=false",
                             fields='files(id,name),nextPageToken', pageToken=tok, pageSize=1000).execute()
        out += r.get('files', []); tok = r.get('nextPageToken')
        if not tok: break
    return out
def up(path, folder, mime='text/csv'):
    svc.files().create(body={'name': os.path.basename(path), 'parents': [folder]},
                       media_body=MediaFileUpload(path, mimetype=mime, resumable=True), fields='id').execute()
    print('  uploaded', os.path.basename(path))
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')

GPU: NVIDIA A100-SXM4-40GB


In [3]:
# Cell 2: fetch the fresh split and images; verify coverage
fid = find_id('eyepacs_split_fresh_v2.csv', F_SPLITS)
dl(fid, 'fresh.csv'); fresh = pd.read_csv('fresh.csv'); fresh['image_id'] = fresh['image_id'].astype(str)
print('fresh split:', len(fresh), 'images |', fresh.patient_id.nunique(), 'patients')

IMG = '/content/fresh'; os.makedirs(IMG, exist_ok=True)
if not glob.glob(f'{IMG}/*.png'):
    for f in sorted(list_folder(F_FRESH_IMG), key=lambda x: x['name']):
        if not f['name'].endswith('.zip'): continue
        z = f"/content/{f['name']}"
        print('downloading', f['name']); dl(f['id'], z)
        with zipfile.ZipFile(z) as zf: zf.extractall(IMG)
        os.remove(z)
INDEX = {os.path.splitext(n)[0]: f'{IMG}/{n}' for n in os.listdir(IMG) if n.endswith('.png')}
missing = sorted(set(fresh['image_id']) - set(INDEX))
print(f'images on disk: {len(INDEX)} | required: {len(fresh)} | missing: {len(missing)}')
if missing: raise SystemExit(f'missing images, e.g. {missing[:5]}')

fresh split: 15794 images | 7897 patients
downloading eyepacs_fresh_batch1_v2.zip
downloading eyepacs_fresh_batch2_v2.zip
downloading eyepacs_fresh_batch3_v2.zip
downloading eyepacs_fresh_batch4_v2.zip
images on disk: 15794 | required: 15794 | missing: 0


In [4]:
# Cell 3: skeletons, strict loading, predict
import timm
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import AutoConfig, AutoModel
from PIL import Image
torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True

TF_EVAL = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])])
class DS(Dataset):
    def __init__(self, df): self.df = df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, i): return TF_EVAL(Image.open(INDEX[self.df.at[i, 'image_id']]).convert('RGB')), i

class RETFound(nn.Module):
    def __init__(self, nc):
        super().__init__()
        cfg = AutoConfig.from_pretrained(RETFOUND_REPO)
        self.base = AutoModel.from_config(cfg, add_pooling_layer=False)
        self.fc_norm = nn.LayerNorm(cfg.hidden_size, eps=1e-6); self.classifier = nn.Linear(cfg.hidden_size, nc)
    def forward(self, x):
        h = self.base(pixel_values=x).last_hidden_state
        return self.classifier(self.fc_norm(h[:, 1:].mean(dim=1)))

def load_model(a, k, dev):
    ck = f'eyepacs_{a}{tag(k)}_v2.pt'
    cid = find_id(ck, F_CHECKPOINTS)
    if not cid: raise FileNotFoundError(ck)
    dl(cid, ck); blob = torch.load(ck, map_location='cpu', weights_only=False); cfg = blob['config']
    m = RETFound(cfg['nc']) if cfg['arch'] == 'retfound' else timm.create_model(cfg['arch'], pretrained=False, num_classes=cfg['nc'])
    m.load_state_dict(blob['state_dict'], strict=True)
    os.remove(ck)
    return m.to(dev).eval(), cfg

@torch.no_grad()
def predict(model, dev, bs):
    dl_ = DataLoader(DS(fresh), batch_size=bs, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    probs = np.zeros(len(fresh), dtype=np.float64)
    for x, idx in dl_:
        out = model(x.to(dev, non_blocking=True)).float()
        p = torch.sigmoid(out[:, 0]) if out.shape[1] == 1 else torch.softmax(out, 1)[:, 2:].sum(1)
        probs[idx.numpy()] = p.double().cpu().numpy()
    return probs
print('ready')

ready


In [5]:
# Cell 4: run all 33 models; write predictions only (no metrics)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
have = {f['name'] for f in list_folder(F_PREDICTIONS)}
todo = [(a, k) for a in ARCHS for k in SEEDS if f'eyepacs_predictions_fresh_{a}{tag(k)}_v2.csv' not in have]
print(f'to run: {len(todo)} of 33')
log = []
for i, (a, k) in enumerate(todo, 1):
    label = a + (tag(k) or '')
    print(f'\n=== [{i}/{len(todo)}] {label} ===')
    t0 = time.time()
    try:
        model, cfg = load_model(a, k, dev)
    except Exception as e:
        print('  FAILED to load:', e); log.append({'model': label, 'status': f'load failed: {e}'}); continue
    pr = predict(model, dev, 32 if cfg['arch'] == 'retfound' else 128)
    out = f'eyepacs_predictions_fresh_{a}{tag(k)}_v2.csv'
    pd.DataFrame({'image_id': fresh['image_id'], 'patient_id': fresh['patient_id'].astype(str), 'eye': fresh['eye'],
                  'dataset': 'eyepacs', 'split': 'fresh',
                  'true_grade': fresh['level'].astype(int), 'true_label': fresh['binary_label'].astype(int),
                  'predicted_probability': pr, 'predicted_class': (pr >= 0.5).astype(int),
                  'model_name': a, 'replicate': k}).to_csv(out, index=False, float_format='%.17g')
    up(out, F_PREDICTIONS)
    log.append({'model': label, 'status': 'ok', 'minutes': round((time.time()-t0)/60, 1),
                'mean_prob': round(float(pr.mean()), 4)})
    print(f"  done in {log[-1]['minutes']} min (mean predicted probability {log[-1]['mean_prob']}; no metrics computed)")
    del model; torch.cuda.empty_cache()

done = {f['name'] for f in list_folder(F_PREDICTIONS)}
got = sum(f'eyepacs_predictions_fresh_{a}{tag(k)}_v2.csv' in done for a in ARCHS for k in SEEDS)
rec = {'generated_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'files_present': got,
       'log': log, 'note': 'Predictions only; no evaluation performed before the v3 lock.'}
with open('fresh_inference_log_v2.json', 'w') as f: json.dump(rec, f, indent=2)
up('fresh_inference_log_v2.json', F_AUDIT, 'application/json')
print(f'\nfresh prediction files: {got}/33')

to run: 33 of 33

=== [1/33] densenet121_binary ===
  uploaded eyepacs_predictions_fresh_densenet121_binary_v2.csv
  done in 0.7 min (mean predicted probability 0.1771; no metrics computed)

=== [2/33] densenet121_binary_seed1 ===
  uploaded eyepacs_predictions_fresh_densenet121_binary_seed1_v2.csv
  done in 0.7 min (mean predicted probability 0.1943; no metrics computed)

=== [3/33] densenet121_binary_seed2 ===
  uploaded eyepacs_predictions_fresh_densenet121_binary_seed2_v2.csv
  done in 0.6 min (mean predicted probability 0.1728; no metrics computed)

=== [4/33] densenet121_5class ===
  uploaded eyepacs_predictions_fresh_densenet121_5class_v2.csv
  done in 0.7 min (mean predicted probability 0.1835; no metrics computed)

=== [5/33] densenet121_5class_seed1 ===
  uploaded eyepacs_predictions_fresh_densenet121_5class_seed1_v2.csv
  done in 0.7 min (mean predicted probability 0.1869; no metrics computed)

=== [6/33] densenet121_5class_seed2 ===
  uploaded eyepacs_predictions_fresh_dens

config.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

  uploaded eyepacs_predictions_fresh_retfound_binary_v2.csv
  done in 1.2 min (mean predicted probability 0.1464; no metrics computed)

=== [26/33] retfound_binary_seed1 ===
  uploaded eyepacs_predictions_fresh_retfound_binary_seed1_v2.csv
  done in 1.1 min (mean predicted probability 0.1595; no metrics computed)

=== [27/33] retfound_binary_seed2 ===
  uploaded eyepacs_predictions_fresh_retfound_binary_seed2_v2.csv
  done in 1.1 min (mean predicted probability 0.1856; no metrics computed)

=== [28/33] retfound_5class ===
  uploaded eyepacs_predictions_fresh_retfound_5class_v2.csv
  done in 1.2 min (mean predicted probability 0.1841; no metrics computed)

=== [29/33] retfound_5class_seed1 ===
  uploaded eyepacs_predictions_fresh_retfound_5class_seed1_v2.csv
  done in 1.1 min (mean predicted probability 0.1736; no metrics computed)

=== [30/33] retfound_5class_seed2 ===
  uploaded eyepacs_predictions_fresh_retfound_5class_seed2_v2.csv
  done in 1.1 min (mean predicted probability 0.1734